In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import pandas as pd
import plotly.io as pio
import polars as pl

from trunx.plot_utils import (
    metric_change_per_plot_tree,
    plot_geographic_location_species,
    plot_histograms_grid,
    plot_social_class_per_plot,
    plot_yearwise_social_class,
)

pio.renderers.default = "notebook"

In [ ]:
tdf = pd.read_pickle("../data/raw/ICP/icpf/03_tidy/icpf-level2_growth-periods_with-cc.pkl.gzip")

tdf = pl.DataFrame(pl.from_pandas(tdf))

print("DataFrame shape:", tdf.shape)

In [ ]:
def dms_to_decimal(dms):
    """Convert DMS packed as ±DDMMSS or ±DDDMMSS to decimal degrees."""
    sign = -1 if str(dms).startswith("-") else 1
    dms = abs(int(dms))

    degrees = dms // 10000
    minutes = (dms % 10000) // 100
    seconds = dms % 100

    val = round(sign * (degrees + minutes / 60 + seconds / 3600), 4)
    return val


tdf = tdf.with_columns(
    pl.col("plot_latitude").map_elements(dms_to_decimal).alias("Lat"),
    pl.col("plot_longitude").map_elements(dms_to_decimal).alias("Lon"),
)

tdf.head()

# Split the data based on species: Spruce, Pine, Beech, Oak

In [ ]:
# "spruce" -> 'Picea abies'
# "pine" -> 'Pinus sylvestris'
# "beech" -> 'Fagus sylvatica'
# "oak" -> 'Quercus robur'/ 'Quercus petraea'

SPECIES_NAME_MAP = {
    "Picea abies": "Spruce",
    "Pinus sylvestris": "Pine",
    "Fagus sylvatica": "Beech",
    "Quercus robur": "Oak",
    "Quercus petraea": "Oak",
}


species = [
    "Picea abies",
    "Pinus sylvestris",
    "Fagus sylvatica",
    "Quercus robur",
    "Quercus petraea",
]

filtered_df = tdf.filter(pl.col("specie").is_in(species))
filtered_df = filtered_df.with_columns(pl.col("specie").replace(SPECIES_NAME_MAP).alias("Species"))

spruce_df = tdf.filter(pl.col("specie") == "Picea abies")
pine_df = tdf.filter(pl.col("specie") == "Pinus sylvestris")
beech_df = tdf.filter(pl.col("specie") == "Fagus sylvatica")
oak_df = tdf.filter(pl.col("specie").is_in(["Quercus robur", "Quercus petraea"]))

print("Filtered DataFrame (Spruce, Pine, Beech, and Oak) shape:", filtered_df.shape, "\n")

print("Spruce")
print("Growth periods:", spruce_df.shape[0])
print("Number of unique trees:", spruce_df.select(pl.col("tree_id")).unique().shape[0])
print("Number of unique plots:", spruce_df.select(pl.col("plot_id")).unique().shape[0], "\n")

print("Pine")
print("Growth periods:", pine_df.shape[0])
print("Number of unique trees:", pine_df.select(pl.col("tree_id")).unique().shape[0])
print("Number of unique plots:", pine_df.select(pl.col("plot_id")).unique().shape[0], "\n")

print("Beech")
print("Growth periods:", beech_df.shape[0])
print("Number of unique trees:", beech_df.select(pl.col("tree_id")).unique().shape[0])
print("Number of unique plots:", beech_df.select(pl.col("plot_id")).unique().shape[0], "\n")

print("Oak")
print("Growth periods:", oak_df.shape[0])
print("Number of unique trees:", oak_df.select(pl.col("tree_id")).unique().shape[0])
print("Number of unique plots:", oak_df.select(pl.col("plot_id")).unique().shape[0], "\n")

## How is social class distributed along the growth periods?

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 6))
ax = plot_yearwise_social_class(tdf, ax, title="All Species")
plt.show()

fig, ax = plt.subplots(2, 2, figsize=(12, 12))

ax[0, 0] = plot_yearwise_social_class(spruce_df, ax[0, 0], title="Spruce")
ax[0, 1] = plot_yearwise_social_class(pine_df, ax[0, 1], title="Pine")
ax[1, 0] = plot_yearwise_social_class(beech_df, ax[1, 0], title="Beech")
ax[1, 1] = plot_yearwise_social_class(oak_df, ax[1, 1], title="Oak")

plt.show()

In [ ]:
plot_social_class_per_plot(
    filtered_df, x="year", y="count", hue="social_class_mode", normalized=True, plot_type="kde"
)

In [ ]:
filtered_df.schema

In [ ]:
metric_change_per_plot_tree(filtered_df)

## Geographic location of different species

In [ ]:
plot_geographic_location_species(filtered_df)

## Number of unique ids

In [ ]:
print("Number of unique plots:", filtered_df["plot_id"].unique().len())

## Number of plots with multiple species

In [ ]:
overlaps = (
    filtered_df.group_by(["Lat", "Lon"])
    .agg(
        pl.col("Species").n_unique().alias("n_species"),
        pl.col("Species").unique().alias("species_list"),
    )
    .filter(pl.col("n_species") > 1)
)

print("Number of plots with multiple species:", overlaps.shape[0])

In [ ]:
social_class_cols = [col for col in filtered_df.schema if col.startswith("social_")]

fig, axes = plot_histograms_grid(
    df=filtered_df,
    columns=social_class_cols,
    hue="Species",
    n_cols=4,
    bins=20,
)

plt.show()

In [ ]:
soil_solution_cols = [col for col in filtered_df.schema if col.startswith("ss_")]

fig, axes = plot_histograms_grid(
    df=filtered_df,
    columns=soil_solution_cols,
    hue="Species",
    n_cols=4,
    bins=20,
)

plt.show()

In [ ]:
dep_cols = [col for col in filtered_df.schema if col.startswith("dep_")]

fig, axes = plot_histograms_grid(
    df=filtered_df,
    columns=dep_cols,
    hue="Species",
    n_cols=4,
    bins=20,
)

plt.show()

In [ ]:
soph_cols = [col for col in filtered_df.schema if col.startswith("soph_")]

fig, axes = plot_histograms_grid(
    df=filtered_df,
    columns=soph_cols,
    hue="Species",
    n_cols=4,
    bins=20,
)
plt.show()

In [ ]:
soil_solution_cols = [col for col in filtered_df.schema if col.startswith("ss_")]
dep_cols = [col for col in filtered_df.schema if col.startswith("dep_")]
soph_cols = [col for col in filtered_df.schema if col.startswith("soph_")]
plot_cols = [col for col in filtered_df.schema if col.startswith("plot_")] + ["Lat", "Lon"]
social_class_cols = [col for col in filtered_df.schema if col.startswith("social_")]
defoliation_cols = [col for col in filtered_df.schema if col.startswith("defoliation_")]
was_cols = [col for col in filtered_df.schema if col.startswith("was_")]

country_cols = [col for col in filtered_df.schema if col.startswith("country")]
growth_cols = [col for col in filtered_df.schema if col.startswith("growth")]
diameter_cols = [col for col in filtered_df.schema if col.startswith("diameter_")]
period_cols = [col for col in filtered_df.schema if col.startswith("period_")]
tree_cols = [col for col in filtered_df.schema if "tree" in col]
other_cols = ["nb_defol_obs", "survey_year"]
specie_cols = ["specie", "Species"]

(
    len(soil_solution_cols)
    + len(dep_cols)
    + len(soph_cols)
    + len(plot_cols)
    + len(social_class_cols)
    + len(defoliation_cols)
    + len(was_cols)
    + len(specie_cols)
    + len(country_cols)
    + len(growth_cols)
    + len(diameter_cols)
    + len(period_cols)
    + len(other_cols)
    + len(tree_cols)
)